In [1]:
import pandas as pd
import json
import plotly.express as px
import planning

In [2]:
# Tennis parameters

CLASSEMENT_JOUEUR = "15/5"

PONDERATION_JOURS = {
    "lundi": 1,
    "mardi": 1,
    "mercredi": 1,
    "jeudi": 1,
    "vendredi": 1,
    "samedi": 1.5,
    "dimanche": 1.5,
    "ferie": 1.5,
}

In [3]:
# Display parameters

TOURNAMENT_COLOR = "grey"
TMC_COLOR = "purple"
PARTICIPATION_COLOR = "royalblue"

DATE_FORMAT = "%d/%m"

TEXT_COLOR = "white"

HOVER_DATA = {
    "debut": False,
    "fin": False,
    "club": False,
    "entree_reelle": True,
    "classement_min": True,
    "classement_max": True,
    "tours_avant_entree": False,
}

In [4]:
with open("tournois.json", encoding="utf-8") as fichier_tournois:
    tournois = json.load(fichier_tournois)


In [5]:
df = pd.DataFrame(tournois)
df = planning.preparer_tournois(
    df,
    CLASSEMENT_JOUEUR,
    PONDERATION_JOURS,
    TOURNAMENT_COLOR,
    TMC_COLOR,
)

df

,club,type,debut,fin,entree_reelle,classement_min,classement_max,couleur_barre,tours_avant_entree,debut_participation
0,L'Haÿ,classique,2026-07-04,2026-07-18,2026-07-12,NC,15/1,grey,7,2026-07-11
1,Fresnes,TMC,2026-07-20,2026-07-21,NaT,15/4,15/1,purple,<NA>,NaT
2,Bourg-la-Reine,classique,2026-07-24,2026-08-15,NaT,NC,4/6,grey,7,2026-07-30


In [6]:
df_classiques = planning.tournois_classiques(df)

fig = px.timeline(
    df,
    x_start="debut",
    x_end="fin",
    y="club",
    hover_data=HOVER_DATA
)
fig.update_traces(marker_color=df["couleur_barre"].tolist())

fig_participation = px.timeline(
    df_classiques,
    x_start="debut_participation",
    x_end="fin",
    y="club",
)
fig_participation.update_traces(marker_color=PARTICIPATION_COLOR)

fig.add_traces(fig_participation.data)

for _, row in df.iterrows():
    debut_label = pd.to_datetime(row["debut"]).strftime(DATE_FORMAT)
    fin_label = pd.to_datetime(row["fin"]).strftime(DATE_FORMAT)
    est_tmc = planning.est_tmc(row["type"])
    tmc_label_x = row["debut"] + (row["fin"] - row["debut"]) / 2
    classement_min = row.get("classement_min")
    classement_max = row.get("classement_max")

    if planning.est_classique(row["type"]) and pd.notna(row["entree_reelle"]):
        fig.add_shape(
            type="line",
            x0=row["entree_reelle"],
            x1=row["entree_reelle"],
            y0=row["club"],
            y1=row["club"],
            y0shift=-0.35,
            y1shift=0.35,
            xref="x",
            yref="y",
            line={"color": "red", "width": 3},
        )

    fig.add_annotation(
        x=tmc_label_x if est_tmc else row["debut"],
        y=row["club"],
        text=f"{debut_label} -> {fin_label}" if est_tmc else debut_label,
        showarrow=False,
        xanchor="center" if est_tmc else "left",
        align="center" if est_tmc else None,
        font={"color": TEXT_COLOR},
        bgcolor="black" if est_tmc else None,
    )

    if not est_tmc:
        fig.add_annotation(
            x=row["fin"],
            y=row["club"],
            text=fin_label,
            showarrow=False,
            xanchor="right",
            font={"color": TEXT_COLOR},
        )

    if planning.est_classique(row["type"]) and pd.notna(row["debut_participation"]):
        fig.add_annotation(
            x=row["debut_participation"],
            y=row["club"],
            text=pd.to_datetime(row["debut_participation"]).strftime(DATE_FORMAT),
            showarrow=False,
            xanchor="left",
            font={"color": TEXT_COLOR},
        )

    if pd.notna(classement_min) and pd.notna(classement_max):
        fig.add_annotation(
            x=row["debut"] + (row["fin"] - row["debut"]) / 2,
            y=row["club"],
            text=f"{classement_min} -> {classement_max}",
            showarrow=False,
            xanchor="center",
            yanchor="top",
            yshift=32,
            font={"color": TEXT_COLOR},
            bgcolor="black",
        )

fig.update_yaxes(autorange="reversed")

# DATE_DEBUT_AFFICHAGE = planning.parse_date_fr("01/07/2026")
# DATE_FIN_AFFICHAGE = planning.parse_date_fr("01/09/2026")
#
# fig.update_xaxes(
#     range=[DATE_DEBUT_AFFICHAGE, DATE_FIN_AFFICHAGE]
# )

fig.show()